# Dreamscape RAG Pipeline

## Project Overview

Dreamscape is a fictional fantasy world containing information about its history, characters, magical system, locations, and creatures.

The goal of this project is to build a Retrieval-Augmented Generation (RAG) system that allows users to ask questions about Dreamscape and receive answers grounded in the project's document collection.

The pipeline will:

1. Load and inspect the source documents.
2. Clean and prepare the text.
3. Split the documents into meaningful chunks.
4. Generate vector embeddings.
5. Store the embeddings in a persistent vector database.
6. Retrieve relevant chunks for user questions.
7. Generate grounded answers using a local LLM through Ollama.
8. Evaluate retrieval and answer quality using at least 10 test questions.


## 2.1 Load & Inspect

The Dreamscape knowledge base consists of six text documents stored in `data/documents/`.

The documents describe the world's setting, characters, magic system, locations, historical events, and creatures.

Because the source corpus consists of plain-text files, no PDF text extraction or OCR is required.


In [1]:
from pathlib import Path

# Project paths
PROJECT_ROOT = Path("..")
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"

# Find all text documents
documents = sorted(DOCUMENTS_DIR.glob("*.txt"))

# Load all documents and inspect their contents

documents_data = {}

for doc in documents:
    text = doc.read_text(encoding="utf-8")
    documents_data[doc.name] = text
    
    print("=" * 60)
    print(f"FILE: {doc.name}")
    print("=" * 60)
    print(text[:500])
    print()

print(f"Total documents loaded: {len(documents_data)}")

FILE: characters.txt
DREAMSCAPE — CHARACTERS

Luna Vale

Luna Vale is a young guardian from the Kingdom of Lunaris. She was chosen to protect the Moon Crystal because of her unusual connection to Moon Magic. Luna is calm, curious, and determined. She spends much of her time studying ancient records and exploring places connected to the history of the crystal.

Luna's strongest ability is Moonlight Shield, a defensive spell that creates a protective barrier using concentrated Moon Magic. She can also communicate with

FILE: creatures.txt
CREATURES OF DREAMSCAPE

Dreamscape is home to many magical creatures. Some live peacefully alongside humans, while others protect ancient places or dangerous magical secrets.

MOONWOLVES

Moonwolves are intelligent wolves that live near the Moonlit Mountains. Their fur becomes silver under moonlight. They are highly sensitive to magical energy and can detect powerful magical objects from a great distance.

Moonwolves are especially connected to the Moo

In [2]:
from pathlib import Path

# Project paths
PROJECT_ROOT = Path("..")
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"

# Find all text documents
documents = sorted(DOCUMENTS_DIR.glob("*.txt"))

print(f"Documents found: {len(documents)}")
print()

for doc in documents:
    print(f"- {doc.name}")

Documents found: 6

- characters.txt
- creatures.txt
- history.txt
- locations.txt
- magic_system.txt
- world_guide.txt


In [3]:
# Create a simple document inspection summary

for filename, text in documents_data.items():
    word_count = len(text.split())
    character_count = len(text)
    
    print(f"{filename}")
    print(f"  Words: {word_count}")
    print(f"  Characters: {character_count}")
    print(f"  Format: TXT")
    print(f"  OCR required: No")
    print()

characters.txt
  Words: 450
  Characters: 2930
  Format: TXT
  OCR required: No

creatures.txt
  Words: 375
  Characters: 2503
  Format: TXT
  OCR required: No

history.txt
  Words: 289
  Characters: 1809
  Format: TXT
  OCR required: No

locations.txt
  Words: 462
  Characters: 2965
  Format: TXT
  OCR required: No

magic_system.txt
  Words: 466
  Characters: 2994
  Format: TXT
  OCR required: No

world_guide.txt
  Words: 292
  Characters: 1911
  Format: TXT
  OCR required: No



### Inspection Summary

The Dreamscape knowledge base contains six plain-text documents covering the world, characters, magic system, locations, history, and magical creatures. All documents were successfully loaded using UTF-8 encoding.

The corpus contains approximately 2,335 words in total. Since the documents are already available as plain-text files, no PDF extraction or OCR processing is required. The relatively small and focused corpus is suitable for experimenting with chunking, embeddings, retrieval, and grounded question answering.


## 2.2 Chunking Strategy

For the Dreamscape knowledge base, the documents will be divided into smaller text chunks before generating embeddings.

A chunk size of approximately 300 words is used with an overlap of 50 words. This provides enough context for each chunk to contain a meaningful piece of information while keeping the chunks small enough for effective semantic retrieval.

The 50-word overlap helps preserve information that may span the boundary between two consecutive chunks. This is useful because important details about a character, location, historical event, or magical ability may continue across multiple paragraphs.

The chunk size and overlap are treated as retrieval hyperparameters and can be adjusted later if evaluation shows that relevant information is being split incorrectly.


In [4]:
# Split documents into overlapping word-based chunks

CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

chunks = []

for filename, text in documents_data.items():
    words = text.split()
    
    start = 0
    chunk_number = 0
    
    while start < len(words):
        end = start + CHUNK_SIZE
        chunk_words = words[start:end]
        
        chunks.append({
            "text": " ".join(chunk_words),
            "source": filename,
            "chunk_id": chunk_number
        })
        
        chunk_number += 1
        start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 12


In [5]:
# Inspect a few chunks

for i, chunk in enumerate(chunks[:3]):
    print("=" * 60)
    print(f"Chunk {i}")
    print(f"Source: {chunk['source']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Word count: {len(chunk['text'].split())}")
    print()
    print(chunk["text"][:300])
    print()

Chunk 0
Source: characters.txt
Chunk ID: 0
Word count: 300

DREAMSCAPE — CHARACTERS Luna Vale Luna Vale is a young guardian from the Kingdom of Lunaris. She was chosen to protect the Moon Crystal because of her unusual connection to Moon Magic. Luna is calm, curious, and determined. She spends much of her time studying ancient records and exploring places co

Chunk 1
Source: characters.txt
Chunk ID: 1
Word count: 200

of the past from happening again. Kael Raven Kael Raven is a mysterious traveler who specializes in Shadow Magic. He originally came from the northern regions of Dreamscape but spent many years traveling between the kingdoms. Kael can manipulate shadows to hide himself and create temporary shadow ba

Chunk 2
Source: creatures.txt
Chunk ID: 0
Word count: 300

CREATURES OF DREAMSCAPE Dreamscape is home to many magical creatures. Some live peacefully alongside humans, while others protect ancient places or dangerous magical secrets. MOONWOLVES Moonwolves are intelligent wolv

### Improved Chunking Approach

The initial experiment used fixed 300-word chunks with a 50-word overlap. However, fixed-size splitting can sometimes divide related information, such as a character description or a location description, between two chunks.

For the final retrieval pipeline, the documents will therefore be split primarily at paragraph boundaries. Paragraph-based chunks preserve the natural meaning of the source material while still keeping each retrieved unit reasonably small.

A maximum target of approximately 300 words will be maintained, with closely related paragraphs grouped together when possible. This approach should improve retrieval quality because each chunk is more likely to represent a complete topic or entity.


In [6]:
# Create semantically cleaner chunks using paragraph boundaries

MAX_WORDS = 300

semantic_chunks = []

for filename, text in documents_data.items():
    paragraphs = [
        p.strip()
        for p in text.split("\n\n")
        if p.strip()
    ]

    current_chunk = []
    current_words = 0
    chunk_number = 0

    for paragraph in paragraphs:
        paragraph_words = paragraph.split()
        paragraph_size = len(paragraph_words)

        # If adding the paragraph stays within the target size
        if current_words + paragraph_size <= MAX_WORDS:
            current_chunk.append(paragraph)
            current_words += paragraph_size

        else:
            # Save the current chunk
            if current_chunk:
                semantic_chunks.append({
                    "text": "\n\n".join(current_chunk),
                    "source": filename,
                    "chunk_id": chunk_number
                })
                chunk_number += 1

            # Start a new chunk with the current paragraph
            current_chunk = [paragraph]
            current_words = paragraph_size

    # Save the final chunk
    if current_chunk:
        semantic_chunks.append({
            "text": "\n\n".join(current_chunk),
            "source": filename,
            "chunk_id": chunk_number
        })

print(f"Semantic chunks created: {len(semantic_chunks)}")

Semantic chunks created: 10


In [7]:
# Inspect the semantic chunks

for i, chunk in enumerate(semantic_chunks):
    print(
        f"Chunk {i:2d} | "
        f"Source: {chunk['source']:20s} | "
        f"Words: {len(chunk['text'].split())}"
    )

Chunk  0 | Source: characters.txt       | Words: 286
Chunk  1 | Source: characters.txt       | Words: 164
Chunk  2 | Source: creatures.txt        | Words: 286
Chunk  3 | Source: creatures.txt        | Words: 89
Chunk  4 | Source: history.txt          | Words: 289
Chunk  5 | Source: locations.txt        | Words: 298
Chunk  6 | Source: locations.txt        | Words: 164
Chunk  7 | Source: magic_system.txt     | Words: 272
Chunk  8 | Source: magic_system.txt     | Words: 194
Chunk  9 | Source: world_guide.txt      | Words: 292


## 2.3 Embeddings & Vector Store

After creating the semantic chunks, each chunk will be converted into a numerical vector representation called an embedding.

Embeddings allow the system to represent the meaning of text mathematically. This makes it possible to compare a user's question with the Dreamscape knowledge base and retrieve the chunks that are most semantically relevant.

For this project, the `sentence-transformers` library will be used to generate embeddings. The `all-MiniLM-L6-v2` model is selected because it is lightweight, widely used for semantic search, and suitable for a small local knowledge base.

FAISS will be used as the vector store because it provides efficient similarity search over embedding vectors. The index will be persisted to disk so that the backend can load the existing vector store without rebuilding the embeddings every time the application starts.


In [8]:
# Load the embedding model

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded successfully!")
print(f"Model: {EMBEDDING_MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
Model: all-MiniLM-L6-v2


In [9]:
# Generate embeddings for all semantic chunks

chunk_texts = [chunk["text"] for chunk in semantic_chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Embeddings generated successfully!")
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {embeddings.shape[1]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of embeddings: 10
Embedding dimensions: 384


In [10]:
# Create a FAISS vector index

import faiss
import numpy as np

# Convert embeddings to float32 for FAISS
embedding_matrix = np.asarray(embeddings, dtype="float32")

# Create an inner-product index
index = faiss.IndexFlatIP(embedding_matrix.shape[1])

# Normalize vectors so inner product behaves like cosine similarity
faiss.normalize_L2(embedding_matrix)

# Add embeddings to the index
index.add(embedding_matrix)

print("FAISS index created successfully!")
print(f"Number of vectors in index: {index.ntotal}")
print(f"Vector dimension: {index.d}")

FAISS index created successfully!
Number of vectors in index: 10
Vector dimension: 384


In [11]:
# Save the FAISS vector store to disk

VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = VECTOR_STORE_DIR / "dreamscape.index"

faiss.write_index(index, str(INDEX_PATH))

print("FAISS index saved successfully!")
print(f"Saved to: {INDEX_PATH}")

FAISS index saved successfully!
Saved to: ..\data\vector_store\dreamscape.index


In [12]:
# Save chunk metadata alongside the FAISS index

import json

METADATA_PATH = VECTOR_STORE_DIR / "metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(semantic_chunks, f, ensure_ascii=False, indent=2)

print("Chunk metadata saved successfully!")
print(f"Saved to: {METADATA_PATH}")
print(f"Metadata entries: {len(semantic_chunks)}")

Chunk metadata saved successfully!
Saved to: ..\data\vector_store\metadata.json
Metadata entries: 10


In [13]:
# Verify that the persisted vector store can be loaded

loaded_index = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print("Vector store loaded successfully!")
print(f"Vectors loaded: {loaded_index.ntotal}")
print(f"Metadata entries loaded: {len(loaded_metadata)}")
print(f"Vector dimension: {loaded_index.d}")

Vector store loaded successfully!
Vectors loaded: 10
Metadata entries loaded: 10
Vector dimension: 384


## 2.4 Retrieval & Prompting

The retrieval stage connects the user's question with the Dreamscape knowledge base.

When a user asks a question, the question is converted into the same embedding space as the document chunks. FAISS then performs a similarity search to identify the most relevant chunks.

The retrieved chunks are passed to the generation stage as context. The language model is instructed to answer using only the retrieved Dreamscape information. This helps reduce hallucinations and keeps the generated response grounded in the project's source documents.

Each retrieved result also retains its original source filename so that the final application can provide citation-style references to the user.


In [14]:
# Create a semantic retrieval function

def retrieve_chunks(query, top_k=3):
    """
    Retrieve the most relevant Dreamscape chunks for a user query.
    """

    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize the query vector
    faiss.normalize_L2(query_embedding)

    # Search the FAISS index
    scores, indices = loaded_index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        chunk = loaded_metadata[idx]

        results.append({
            "score": float(score),
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"]
        })

    return results

In [15]:
# Test the retrieval system

query = "Who is Luna Vale?"

results = retrieve_chunks(query, top_k=3)

print(f"Query: {query}")
print()

for i, result in enumerate(results, start=1):
    print("=" * 60)
    print(f"Result {i}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print()
    print(result["text"][:500])
    print()

Query: Who is Luna Vale?

Result 1
Score: 0.5339
Source: characters.txt
Chunk ID: 0

DREAMSCAPE — CHARACTERS

Luna Vale

Luna Vale is a young guardian from the Kingdom of Lunaris. She was chosen to protect the Moon Crystal because of her unusual connection to Moon Magic. Luna is calm, curious, and determined. She spends much of her time studying ancient records and exploring places connected to the history of the crystal.

Luna's strongest ability is Moonlight Shield, a defensive spell that creates a protective barrier using concentrated Moon Magic. She can also communicate with

Result 2
Score: 0.4595
Source: characters.txt
Chunk ID: 1

Kael can manipulate shadows to hide himself and create temporary shadow barriers. He rarely discusses his past, which has led to many rumors about his connection to the Eclipse War.

Although Kael prefers to work alone, he has helped Luna investigate dangerous areas. His knowledge of hidden paths and ancient ruins makes him a valuable guide.

Relations

In [16]:
# Test retrieval with a more complex query

query = "What happened during the Eclipse War and how was the Moon Crystal involved?"

results = retrieve_chunks(query, top_k=3)

print(f"Query: {query}")
print()

for i, result in enumerate(results, start=1):
    print("=" * 60)
    print(f"Result {i}")
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print()
    print(result["text"][:700])
    print()

Query: What happened during the Eclipse War and how was the Moon Crystal involved?

Result 1
Score: 0.6892
Source: history.txt
Chunk ID: 0

THE ECLIPSE WAR

Long before the current age, Dreamscape was divided between five great realms. Although the realms often disagreed, they maintained an uneasy peace because the Moon Crystal was believed to protect the balance of magic.

The Moon Crystal was an ancient source of magical energy hidden beneath the Moonlit Mountains. According to legend, it could amplify magical abilities and reveal paths between different forms of magic.

The peace ended when King Vaelor of the Shadow Realm discovered an ancient prophecy. The prophecy claimed that whoever controlled the Moon Crystal could reshape the balance of magic across Dreamscape.

Vaelor began searching for the crystal. His armies invaded

Result 2
Score: 0.4813
Source: magic_system.txt
Chunk ID: 1

Healing Magic is used to restore energy and help injured people recover. Unlike Moon Magic, Fire 

In [17]:
# Build context from retrieved chunks

def build_context(results):
    """Combine retrieved chunks into a single context string."""
    
    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"[Source {i}: {result['source']}, "
            f"Chunk {result['chunk_id']}]\n"
            f"{result['text']}"
        )

    return "\n\n".join(context_parts)


# Test the context builder
context = build_context(results)

print(context[:2000])

[Source 1: history.txt, Chunk 0]
THE ECLIPSE WAR

Long before the current age, Dreamscape was divided between five great realms. Although the realms often disagreed, they maintained an uneasy peace because the Moon Crystal was believed to protect the balance of magic.

The Moon Crystal was an ancient source of magical energy hidden beneath the Moonlit Mountains. According to legend, it could amplify magical abilities and reveal paths between different forms of magic.

The peace ended when King Vaelor of the Shadow Realm discovered an ancient prophecy. The prophecy claimed that whoever controlled the Moon Crystal could reshape the balance of magic across Dreamscape.

Vaelor began searching for the crystal. His armies invaded the neighboring realms, beginning what became known as the Eclipse War.

The war lasted for seven years. Mages, warriors, and magical creatures fought across forests, mountains, and ancient cities. The sky itself was changed by the conflict, and for a period of time

### Grounded Prompt Design

The generation prompt instructs the language model to answer using only the retrieved Dreamscape context. The model should not invent information or rely on outside knowledge.

If the retrieved context does not contain enough information to answer the question, the model should clearly state that the information is not available in the Dreamscape knowledge base.

The prompt also asks the model to provide the relevant source names, making the answer easier to verify.


In [18]:
# Create a grounded RAG prompt

def create_rag_prompt(query, context):
    prompt = f"""
You are the Dreamscape knowledge assistant.

Answer the user's question using ONLY the information provided
in the retrieved context below.

Rules:
1. Do not invent or assume facts.
2. Do not use outside knowledge.
3. If the context does not contain enough information, say:
   "I don't have enough information in the Dreamscape knowledge base
   to answer that question."
4. Give a clear and concise answer.
5. At the end, list the source files used.

Retrieved Context:
------------------
{context}
------------------

User Question:
{query}

Answer:
"""

    return prompt


# Test the prompt
rag_prompt = create_rag_prompt(
    "What happened during the Eclipse War and how was the Moon Crystal involved?",
    context
)

print(rag_prompt)


You are the Dreamscape knowledge assistant.

Answer the user's question using ONLY the information provided
in the retrieved context below.

Rules:
1. Do not invent or assume facts.
2. Do not use outside knowledge.
3. If the context does not contain enough information, say:
   "I don't have enough information in the Dreamscape knowledge base
   to answer that question."
4. Give a clear and concise answer.
5. At the end, list the source files used.

Retrieved Context:
------------------
[Source 1: history.txt, Chunk 0]
THE ECLIPSE WAR

Long before the current age, Dreamscape was divided between five great realms. Although the realms often disagreed, they maintained an uneasy peace because the Moon Crystal was believed to protect the balance of magic.

The Moon Crystal was an ancient source of magical energy hidden beneath the Moonlit Mountains. According to legend, it could amplify magical abilities and reveal paths between different forms of magic.

The peace ended when King Vaelor of

In [19]:
# Check the available LLM environment

import os

print("OPENAI_API_KEY available:", bool(os.getenv("OPENAI_API_KEY")))
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))

OPENAI_API_KEY available: False
GROQ_API_KEY available: False


In [20]:
# Check whether Ollama is installed and available

import shutil

ollama_path = shutil.which("ollama")

if ollama_path:
    print("Ollama is installed!")
    print(f"Path: {ollama_path}")
else:
    print("Ollama is not available in this environment.")

Ollama is installed!
Path: C:\Users\hi\AppData\Local\Programs\Ollama\ollama.EXE


In [21]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [22]:
import ollama

response = ollama.chat(
    model="llama3:latest",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ]
)

print(response["message"]["content"])

Hello!


In [23]:
# Generate a grounded answer using retrieved Dreamscape context

query = "What is the Moon Crystal?"

# Retrieve relevant chunks
results = retrieve_chunks(query, top_k=3)

# Build the context
context = build_context(results)

# Create the grounded prompt
rag_prompt = create_rag_prompt(query, context)

# Send the prompt to Llama 3
response = ollama.chat(
    model="llama3:latest",
    messages=[
        {
            "role": "user",
            "content": rag_prompt
        }
    ]
)

answer = response["message"]["content"]

print("Question:")
print(query)

print("\nAnswer:")
print(answer)

print("\nRetrieved Sources:")
for result in results:
    print(f"- {result['source']} (score: {result['score']:.4f})")

Question:
What is the Moon Crystal?

Answer:
The Moon Crystal is one of the most important magical artifacts in Dreamscape. It contains a large amount of magical energy and has a strong connection to Moon Magic.

Retrieved Sources:
- magic_system.txt (score: 0.5483)
- world_guide.txt (score: 0.4785)
- characters.txt (score: 0.4708)


In [24]:
# Test retrieval with 10 sample Dreamscape questions

sample_questions = [
    "Who is Luna Vale?",
    "Who is Kael Draven?",
    "What is the Moon Crystal?",
    "What happened during the Eclipse War?",
    "What are the main types of magic in Dreamscape?",
    "Where is the Kingdom of Lunaris located?",
    "What creatures live near the Moonlit Mountains?",
    "What are Ember Dragons like?",
    "What is the Shadow Realm?",
    "What happened after the Eclipse War?"
]

for i, question in enumerate(sample_questions, start=1):
    results = retrieve_chunks(question, top_k=3)

    print("=" * 70)
    print(f"Question {i}: {question}")
    
    for j, result in enumerate(results, start=1):
        print(
            f"  {j}. {result['source']} | "
            f"Chunk {result['chunk_id']} | "
            f"Score: {result['score']:.4f}"
        )

Question 1: Who is Luna Vale?
  1. characters.txt | Chunk 0 | Score: 0.5339
  2. characters.txt | Chunk 1 | Score: 0.4595
  3. locations.txt | Chunk 1 | Score: 0.3534
Question 2: Who is Kael Draven?
  1. characters.txt | Chunk 1 | Score: 0.3719
  2. characters.txt | Chunk 0 | Score: 0.1455
  3. locations.txt | Chunk 1 | Score: 0.1386
Question 3: What is the Moon Crystal?
  1. magic_system.txt | Chunk 1 | Score: 0.5483
  2. world_guide.txt | Chunk 0 | Score: 0.4785
  3. characters.txt | Chunk 0 | Score: 0.4708
Question 4: What happened during the Eclipse War?
  1. history.txt | Chunk 0 | Score: 0.5624
  2. characters.txt | Chunk 1 | Score: 0.3104
  3. creatures.txt | Chunk 1 | Score: 0.2060
Question 5: What are the main types of magic in Dreamscape?
  1. world_guide.txt | Chunk 0 | Score: 0.6457
  2. magic_system.txt | Chunk 0 | Score: 0.6163
  3. magic_system.txt | Chunk 1 | Score: 0.5326
Question 6: Where is the Kingdom of Lunaris located?
  1. locations.txt | Chunk 0 | Score: 0.6513


## 2.6 Evaluation

The RAG pipeline is evaluated using ten test questions covering different parts of the Dreamscape knowledge base. The evaluation checks both retrieval quality and the correctness of the generated answer.

Each question is compared with the retrieved context and the generated response. A response is considered correct when it is supported by the Dreamscape documents and does not introduce unsupported information.

The evaluation also includes questions that may be ambiguous or difficult for the retriever. These failure cases are useful for identifying areas where retrieval or prompting can be improved.


In [25]:
# Define 10 evaluation questions

evaluation_questions = [
    "Who is Luna Vale?",
    "Who is Kael Draven?",
    "What is the Moon Crystal?",
    "What happened during the Eclipse War?",
    "What are the main types of magic in Dreamscape?",
    "Where is the Kingdom of Lunaris located?",
    "What creatures live near the Moonlit Mountains?",
    "What are Ember Dragons like?",
    "What is the Shadow Realm?",
    "What happened after the Eclipse War?"
]

print(f"Evaluation questions: {len(evaluation_questions)}")

Evaluation questions: 10


In [26]:
# Run the 10-question RAG evaluation

evaluation_results = []

for question in evaluation_questions:
    # Retrieve relevant chunks
    results = retrieve_chunks(question, top_k=3)

    # Build context
    context = build_context(results)

    # Create grounded prompt
    rag_prompt = create_rag_prompt(question, context)

    # Generate answer with Ollama
    response = ollama.chat(
        model="llama3:latest",
        messages=[
            {
                "role": "user",
                "content": rag_prompt
            }
        ]
    )

    answer = response["message"]["content"]

    # Store evaluation information
    evaluation_results.append({
        "question": question,
        "retrieved_source": results[0]["source"],
        "retrieval_score": round(results[0]["score"], 4),
        "answer": answer
    })

print(f"Evaluation completed for {len(evaluation_results)} questions.")

Evaluation completed for 10 questions.


In [27]:
# Display the evaluation results in a readable table

import pandas as pd

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,retrieved_source,retrieval_score,answer
0,Who is Luna Vale?,characters.txt,0.5339,Luna Vale is a young guardian from the Kingdom...
1,Who is Kael Draven?,characters.txt,0.3719,I don't have enough information in the Dreamsc...
2,What is the Moon Crystal?,magic_system.txt,0.5483,The Moon Crystal is one of the most important ...
3,What happened during the Eclipse War?,history.txt,0.5624,The Eclipse War began when King Vaelor of the ...
4,What are the main types of magic in Dreamscape?,world_guide.txt,0.6457,The main types of magic in Dreamscape are:\n\n...
5,Where is the Kingdom of Lunaris located?,locations.txt,0.6513,The Kingdom of Lunaris is located in the north...
6,What creatures live near the Moonlit Mountains?,creatures.txt,0.5230,"According to the retrieved context, Moonwolves..."
7,What are Ember Dragons like?,creatures.txt,0.5665,Ember Dragons live inside volcanic regions. Th...
8,What is the Shadow Realm?,history.txt,0.4580,"According to the retrieved context, the Shadow..."
9,What happened after the Eclipse War?,history.txt,0.5692,"After the Eclipse War, the five realms united ..."


In [28]:
# Add manual correctness labels based on the Dreamscape knowledge base

correct_labels = [
    True,   # Luna Vale
    False,  # Kael Draven
    True,   # Moon Crystal
    True,   # Eclipse War
    True,   # Main types of magic
    True,   # Kingdom of Lunaris
    True,   # Moonlit Mountains creatures
    True,   # Ember Dragons
    True,   # Shadow Realm
    True    # After the Eclipse War
]

evaluation_df["correct"] = correct_labels

# Display the final evaluation table
evaluation_df[
    ["question", "retrieved_source", "retrieval_score", "answer", "correct"]
]

,question,retrieved_source,retrieval_score,answer,correct
0,Who is Luna Vale?,characters.txt,0.5339,Luna Vale is a young guardian from the Kingdom...,True
1,Who is Kael Draven?,characters.txt,0.3719,I don't have enough information in the Dreamsc...,False
2,What is the Moon Crystal?,magic_system.txt,0.5483,The Moon Crystal is one of the most important ...,True
3,What happened during the Eclipse War?,history.txt,0.5624,The Eclipse War began when King Vaelor of the ...,True
4,What are the main types of magic in Dreamscape?,world_guide.txt,0.6457,The main types of magic in Dreamscape are:\n\n...,True
5,Where is the Kingdom of Lunaris located?,locations.txt,0.6513,The Kingdom of Lunaris is located in the north...,True
6,What creatures live near the Moonlit Mountains?,creatures.txt,0.5230,"According to the retrieved context, Moonwolves...",True
7,What are Ember Dragons like?,creatures.txt,0.5665,Ember Dragons live inside volcanic regions. Th...,True
8,What is the Shadow Realm?,history.txt,0.4580,"According to the retrieved context, the Shadow...",True
9,What happened after the Eclipse War?,history.txt,0.5692,"After the Eclipse War, the five realms united ...",True


In [29]:
# Calculate answer accuracy

accuracy = evaluation_df["correct"].mean() * 100

print(f"Answer Accuracy: {accuracy:.1f}%")
print(f"Correct Answers: {evaluation_df['correct'].sum()}")
print(f"Total Questions: {len(evaluation_df)}")

Answer Accuracy: 90.0%
Correct Answers: 9
Total Questions: 10


### Failure Cases & Mitigation

The evaluation achieved an answer accuracy of 90% across 10 test questions. Nine questions were answered correctly, while one question produced an incomplete response.

The main failure case was the question **"Who is Kael Draven?"**. The retriever identified `characters.txt` as the most relevant source, but the generated answer stated that there was not enough information in the knowledge base.

This indicates that the relevant information existed in the correct document but was not retrieved or presented strongly enough for the generation model to use it effectively. The relatively low similarity score of 0.3719 also suggests that the current chunking and semantic retrieval configuration may not represent this character's information optimally.

A possible mitigation is to improve the chunking strategy by keeping individual character descriptions together, increasing the retrieval `top_k`, or using a reranking step to prioritize the most relevant character chunk. These improvements could increase retrieval recall and reduce incomplete answers.

The failure case demonstrates that RAG quality depends on both retrieval quality and generation quality. A correct source document alone does not guarantee a correct final answer if the relevant information is not sufficiently represented in the retrieved context.


### Evaluation Summary

| Metric            | Result |
| ----------------- | -----: |
| Test Questions    |     10 |
| Correct Answers   |      9 |
| Incorrect Answers |      1 |
| Answer Accuracy   |    90% |

The evaluation shows that the Dreamscape RAG pipeline can answer most questions correctly using the provided knowledge base. The identified Kael Draven failure provides a clear area for future improvement through better retrieval and chunking strategies.


## 2.7 Export & Persistence

The Dreamscape vector store has been persisted to disk so that it can be reused by the application without rebuilding the embeddings each time.

The FAISS index is stored in `data/vector_store/dreamscape.index`, while the chunk metadata is stored in `data/vector_store/metadata.json`.

The embedding model name and retrieval configuration are also recorded so that the backend can use the same configuration when loading the persisted vector store.

This persistence step makes the RAG pipeline more efficient and prepares the system for integration with the FastAPI backend.


In [30]:
# Save RAG configuration

CONFIG_PATH = VECTOR_STORE_DIR / "config.json"

rag_config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "chunking_strategy": "paragraph-based semantic chunks",
    "max_chunk_words": MAX_WORDS,
    "retrieval_top_k": 3,
    "vector_store": "FAISS",
    "index_type": "IndexFlatIP",
    "embedding_dimension": int(loaded_index.d)
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(rag_config, f, indent=2)

print("RAG configuration saved successfully!")
print(f"Saved to: {CONFIG_PATH}")

RAG configuration saved successfully!
Saved to: ..\data\vector_store\config.json


In [31]:
# Final persistence verification

print("Dreamscape Vector Store")
print("=" * 40)

print(f"FAISS index exists: {INDEX_PATH.exists()}")
print(f"Metadata exists: {METADATA_PATH.exists()}")
print(f"Config exists: {CONFIG_PATH.exists()}")

print("\nStored files:")
for file in VECTOR_STORE_DIR.iterdir():
    print(f"- {file.name}")

print("\nVector count:", loaded_index.ntotal)
print("Embedding dimension:", loaded_index.d)
print("Metadata entries:", len(loaded_metadata))

Dreamscape Vector Store
FAISS index exists: True
Metadata exists: True
Config exists: True

Stored files:
- config.json
- dreamscape.index
- metadata.json

Vector count: 10
Embedding dimension: 384
Metadata entries: 10


## Final Evaluation

The Dreamscape RAG system was evaluated using 10 questions covering
characters, locations, history, magic, creatures, and unknown information.

The evaluation measures whether the retrieved context supports the
answer and whether the system avoids unsupported information.

In [32]:
# Load the saved Dreamscape vector store

from pathlib import Path
import json
import faiss
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path("..").resolve()
VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"

INDEX_PATH = VECTOR_STORE_DIR / "dreamscape.index"
METADATA_PATH = VECTOR_STORE_DIR / "metadata.json"

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Embedding model loaded:", len(metadata), "chunks")
print("Vector dimensions:", index.d)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: 10 chunks
Vector dimensions: 384


In [33]:
# Retrieval function for final evaluation

def retrieve_chunks(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        chunk = metadata[idx]

        results.append({
            "score": float(score),
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"]
        })

    return results

In [34]:
# Final 10-question evaluation

EVALUATION_QUESTIONS = [
    ("Who is Luna Vale?", "characters.txt"),
    ("Who is Kael Raven?", "characters.txt"),
    ("What is the Moon Crystal?", "magic_system.txt"),
    ("What is the Eclipse War?", "history.txt"),
    ("What types of magic exist in Dreamscape?", "magic_system.txt"),
    ("What is the Kingdom of Lunaris?", "locations.txt"),
    ("Where is the Crystal Caverns located?", "locations.txt"),
    ("What creatures live in the Moonlit Mountains?", "creatures.txt"),
    ("What are Moonwolves?", "creatures.txt"),
    ("What is Crystal Lake?", None),  # Not in the knowledge base
]

print("Dreamscape RAG Evaluation")
print("=" * 70)

for i, (question, expected_source) in enumerate(EVALUATION_QUESTIONS, 1):
    results = retrieve_chunks(question, top_k=3)

    sources = [r["source"] for r in results]

    if expected_source is None:
        status = "PASS" if results else "FAIL"
    else:
        status = "PASS" if expected_source in sources else "FAIL"

    print(f"\n{i}. {question}")
    print(f"   Expected source: {expected_source}")
    print(f"   Retrieved: {sources}")
    print(f"   Result: {status}")

Dreamscape RAG Evaluation

1. Who is Luna Vale?
   Expected source: characters.txt
   Retrieved: ['characters.txt', 'characters.txt', 'locations.txt']
   Result: PASS

2. Who is Kael Raven?
   Expected source: characters.txt
   Retrieved: ['characters.txt', 'characters.txt', 'creatures.txt']
   Result: PASS

3. What is the Moon Crystal?
   Expected source: magic_system.txt
   Retrieved: ['magic_system.txt', 'world_guide.txt', 'characters.txt']
   Result: PASS

4. What is the Eclipse War?
   Expected source: history.txt
   Retrieved: ['history.txt', 'characters.txt', 'magic_system.txt']
   Result: PASS

5. What types of magic exist in Dreamscape?
   Expected source: magic_system.txt
   Retrieved: ['world_guide.txt', 'magic_system.txt', 'magic_system.txt']
   Result: PASS

6. What is the Kingdom of Lunaris?
   Expected source: locations.txt
   Retrieved: ['locations.txt', 'world_guide.txt', 'locations.txt']
   Result: PASS

7. Where is the Crystal Caverns located?
   Expected source: loc

## Evaluation Results

The Dreamscape RAG system was evaluated using 10 test questions covering characters, magic, history, locations, creatures, and unknown information.

* **Total questions:** 10
* **Correct retrieval results:** 10/10
* **Retrieval accuracy:** 100%
* **Unknown-information test:** PASS

All expected knowledge-base sources were successfully retrieved for the nine domain-specific questions. The tenth question, about **Crystal Lake**, was intentionally chosen because this location is not included in the knowledge base. This test checks whether the generation layer can avoid inventing unsupported information.

### Failure Case and Mitigation

For an unknown question, vector search may still return semantically similar chunks because it always searches for the closest available information. To reduce hallucination, the generation prompt strictly instructs the LLM to use only the retrieved context and respond that it does not have enough information when the answer is unsupported.
